In [ ]:
from pathlib import Path
import os,sys,json,hashlib,importlib.metadata,time
OWN=Path.cwd()/"llm4rec_s2_rq2"
os.environ["CUDA_VISIBLE_DEVICES"]=""
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
os.environ["USE_TF"]="0"
os.environ["USE_FLAX"]="0"
sys.path.insert(0,str(OWN/"offline_packages_v1"))
shared_path=OWN/"shared_contexts"
contexts=[json.loads(l) for l in (shared_path/"contexts.jsonl").read_text().splitlines()]
ours=[json.loads(l) for l in (OWN/"baseline_seed0/contexts.jsonl").read_text().splitlines()]
protocol=json.loads((shared_path/"protocol.json").read_text())
assert protocol==json.loads((OWN/"baseline_seed0/protocol.json").read_text())
assert protocol["protocol_hash"]=="50efd4aeb415ba6f0c66dd0793a7c2033cdc32a42b86b58eebbca3e81eaf3417"
assert [c["user_id"] for c in contexts]==[c["user_id"] for c in ours]==protocol["users"]
changed=[]
for a,b in zip(contexts,ours):
    assert len(a["candidate_assets"])==len(b["candidate_assets"])==50
    if a["candidate_assets"]!=b["candidate_assets"]:changed.append(a["user_id"])
    normalized_a=dict(a,candidate_assets=sorted(a["candidate_assets"],key=lambda x:x["id"]))
    normalized_b=dict(b,candidate_assets=sorted(b["candidate_assets"],key=lambda x:x["id"]))
    assert normalized_a==normalized_b
assert len(changed)==3
hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in shared_path.iterdir() if p.is_file()}
assert hashes["contexts.jsonl"]=="a505745968316fb96f03a453726878d84501965a12482e81b54b8bf468a1093e"
parity={"users":100,"histories_equal":True,"candidate_sets_equal":True,"all_fields_equal_after_candidate_sort":True,"candidate_order_different_users":changed,"adopted_input":"shared_contexts/contexts.jsonl (identical ordered snapshot for B and C)","source_is_shared_input_only_no_teammate_model_outputs":True,"sha256":hashes,"provenance":json.loads((shared_path/"provenance.json").read_text())}
(OWN/"shared-context-parity.json").write_text(json.dumps(parity,indent=2)+"\n")
import torch,transformers
import transformers.utils.import_utils as import_utils
optional_flags={}
for flag in ["_librosa_available","_torchvision_available","_torchaudio_available"]:
    optional_flags[flag]={"before":getattr(import_utils,flag,None),"after":False}
    setattr(import_utils,flag,False)
# Qwen3 is text only. Disable optional audio/vision discovery in this process, no package edits.
from transformers import AutoTokenizer,AutoModelForCausalLM
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
assert transformers.__version__=="4.57.6"
assert torch.__version__=="2.9.1+cu128"
versions={d.metadata["Name"]:d.version for d in importlib.metadata.distributions(path=[str(OWN/"offline_packages_v1")])}
for package in ["torch","triton","sympy","networkx","jinja2","numpy","nvidia-cuda-runtime-cu12","nvidia-cublas-cu12","nvidia-cudnn-cu12"]:
    try:versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:versions[package]=None
meta={"imports_passed":True,"GPU_allocated":False,"python":sys.version,"torch":torch.__version__,"transformers":transformers.__version__,"huggingface-hub":importlib.metadata.version("huggingface-hub"),"versions":versions,"dependency_paths":{"torch":torch.__file__,"transformers":transformers.__file__},"optional_audio_vision_discovery_disabled_process_only":optional_flags,"torch_deviation":"Existing working CUDA torch2.9.1+cu128 retained by user authorization instead of teammates torch2.11 CPU; no teammate model outputs used."}
(OWN/"offline-import-validation.json").write_text(json.dumps(meta,indent=2)+"\n")
(OWN/"requirements-inference-actual.txt").write_text("".join(k+"=="+str(v)+"\n" for k,v in sorted(versions.items())))
print(json.dumps({"parity":parity,"imports":meta},indent=2))
